<a href="https://colab.research.google.com/github/Dimuthurathnayake668/signinform/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. පරණ කුණු සේරම අයින් කරමු
!pip uninstall unsloth xformers trl peft accelerate bitsandbytes transformers datasets -y

# 2. Unsloth සහ Zoo දාමු
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

# 3. මෙන්න මේවා තමයි උඹේ ERROR එකෙන් ඉල්ලන නිවැරදිම පරාසය (Constraints)
# Transformers 5.2.0 (Constraint <= 5.2.0)
# TRL 0.24.0 (Constraint <= 0.24.0)
!pip install transformers==5.2.0 \
             trl==0.24.0 \
             datasets==3.6.0 \
             xformers==0.0.35 \
             peft==0.18.1 \
             accelerate==1.4.0 \
             bitsandbytes==0.49.2 \
             cut_cross_entropy \
             hf_transfer \
             torchao==0.13.0

Found existing installation: unsloth 2026.3.4
Uninstalling unsloth-2026.3.4:
  Successfully uninstalled unsloth-2026.3.4
Found existing installation: xformers 0.0.35
Uninstalling xformers-0.0.35:
  Successfully uninstalled xformers-0.0.35
Found existing installation: trl 0.29.0
Uninstalling trl-0.29.0:
  Successfully uninstalled trl-0.29.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: transformers 5.3.0
Uninstalling transformers-5.3.0:
  Successfully uninstalled transformers-5.3.0
Found existing installation: datasets 4.3.0
Uninstalling datasets-4.3.0:
  Successfully uninstalled datasets-4.3.0
  Cloning https://github.com/unslothai/unsloth.git to /t

In [3]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# 1. මොඩල් එක Load කරමු
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/meta-llama-3.1-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. LoRA Adapters Injection (Rank 32 for Trading Logic)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

# 3. Dataset Preparation
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    reasonings   = examples["reasoning"]
    outputs      = examples["output"]
    texts = []
    for i, r, o in zip(instructions, reasonings, outputs):
        text = f"MARKET_DATA: {i}\nSTRATEGY_LOGIC: {r}\nACTION: {o}{tokenizer.eos_token}"
        texts.append(text)
    return { "text" : texts }

dataset = load_dataset("json", data_files={"train": "predator_v3_1_5000.jsonl"}, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 4. Training Configuration (Strictly compatible with TRL 0.24.0)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = SFTConfig(
        output_dir = "predator_run",
        max_steps = 320,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        learning_rate = 5e-5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
    ),
)

print("PREDATOR SYSTEM IS ONLINE. STARTING INJECTION...")
trainer.train()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
PREDATOR SYSTEM IS ONLINE. STARTING INJECTION...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 2 | Total steps = 320
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
1,3.820455
2,3.853187
3,3.868427
4,3.815875
5,3.864452
6,3.779792
7,3.819688
8,3.699948
9,3.678784
10,3.565037


KeyboardInterrupt: 

In [4]:
model.save_pretrained_gguf("predator_final_v3", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:49<00:00, 72.25s/it]


Unsloth: Merge process complete. Saved to `/content/predator_final_v3`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['predator_final_v3_gguf/Meta-Llama-3.1-8B.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['predator_final_v3_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/Meta-Llama-3.1-8B'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model predator_final_v3_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'predator_final_v3',
 'gguf_directory': 'predator_final_v3_gguf',
 'gguf_files': ['predator_final_v3_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [7]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("predator_final_v3_gguf/Meta-Llama-3.1-8B.Q4_K_M.gguf", "/content/drive/MyDrive/")


Mounted at /content/drive


'/content/drive/MyDrive/Meta-Llama-3.1-8B.Q4_K_M.gguf'

In [8]:
from unsloth import FastLanguageModel
import torch

# 1. මොඩල් එක Inference (ප්‍රතිචාර ලබාගන්නා) මාදිලියට හරවමු
FastLanguageModel.for_inference(model)

# 2. උඹේ මාර්කට් දත්ත මෙතනට දාපන්
# උදාහරණ: RSI වැඩියි, Trend එක Overbought වගේ එකක්
market_data = "BTC/USDT, Price: 69200, RSI: 82, Trend: Strong Bullish, Volume: Neutral"

prompt = f"MARKET_DATA: {market_data}\nSTRATEGY_LOGIC: "

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Predator ගෙන් තීරණය ඉල්ලමු
print("--- PREDATOR IS ANALYZING ---")
outputs = model.generate(
    **inputs,
    max_new_tokens = 150,
    use_cache = True,
    pad_token_id = tokenizer.eos_token_id
)

# 4. පිළිතුර මුද්‍රණය කරමු
response = tokenizer.decode(outputs[0], skip_special_tokens = True)
print(response)

--- PREDATOR IS ANALYZING ---


--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

MARKET_DATA: BTC/USDT, Price: 69200, RSI: 82, Trend: Strong Bullish, Volume: Neutral
STRATEGY_LOGIC: 82 RSI is extremely overbought. Aggressive SHORT is highly probable. Maintain capital safety by implementing strict SLIPPAGE_PROTECTION of 2.0% | 1400 PIPS.
ACTION: IDLE | CAPITAL_SAFETY: 100% | SLIPPAGE_PROTECTION: 2.0% | CONFIDENCE: 92% | SLIPPAGE: 0.0%


In [12]:
from unsloth import FastLanguageModel

# මෙතන "username/model_name" වෙනුවට උඹේ HF username එක දාපන්
# උදා: "your_name/predator-v3-gguf"
model_name = "dimuth/predator-llama-3.1-gguf"

model.push_to_hub_gguf(
    model_name,
    tokenizer,
    quantization_method = "q4_k_m",
    token = True # උඩදී ලොග් වුණ නිසා මේක True දාන්න
)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [06:47<00:00, 101.97s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:14<00:00, 63.51s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_2fq3ay80`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...


KeyboardInterrupt: 

In [11]:
from huggingface_hub import login
login()